<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Efecto de la ganancia exportadora sobre el producto interno bruto

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `typing`: anotaciones de tipo de la función de remuestreo.

In [1]:
import pandas as pd
import numpy as np
from typing import Optional

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp, processed, tables

### Funciones auxiliares

Define el formato de presentación en pesos y el remuestreo por porcentaje que usa el bootstrap.

In [3]:
def formato_pesos(valor):
    return f"${valor:,.0f} COP"

def formato_usd(valor):
    return f"${valor:,.0f} USD"

def resample_percent_df(
    data: pd.DataFrame,
    percent: float,
    replace: bool = True,
    random_state: Optional[int] = None
) -> pd.DataFrame:
    """
    Toma un x% de las filas de un DataFrame vía remuestreo aleatorio uniforme.

    Parámetros
    ----------
    data : pd.DataFrame
        DataFrame original.
    percent : float
        Porcentaje deseado (ej. 33.3 para 33.3%).
        Si replace=True puede ser >100 (muestras mayores que n).
    replace : bool, por defecto True
        True: con reemplazo (bootstrap). False: sin reemplazo.
    random_state : int, opcional
        Semilla para reproducibilidad.
    return_index : bool, por defecto False
        Si True, retorna también los índices posicionales seleccionados.

    Retorna
    -------
    pd.DataFrame  o  (pd.DataFrame, np.ndarray)
        Submuestra (y opcionalmente los índices posicionales).
    """
    if not isinstance(data, pd.DataFrame):
        raise TypeError("`data` debe ser un pandas.DataFrame.")

    n = len(data)
    if n == 0:
        raise ValueError("El DataFrame está vacío.")
    if percent <= 0:
        raise ValueError("`percent` debe ser > 0.")

    # Tamaño objetivo
    k = int(np.round(n * (percent / 100.0)))
    k = max(k, 1)
    if not replace and k > n:
        k = n

    rng = np.random.default_rng(random_state)
    idx_pos = rng.choice(n, size=k, replace=replace)

    out = data.iloc[idx_pos]  # conserva los labels de índice originales
    return out

### Lectura de datos

Carga los multiplicadores de Leontief, la correspondencia entre la CIIU y las ramas de la matriz
insumo-producto, las predicciones de ganancia exportadora y la participación del valor agregado
en la producción de cada rama.

In [4]:
df_multiplicadores = pd.read_parquet(temp / 'pib' / 'multiplicadores_leontief.parquet')

df_ciiu = pd.read_parquet(temp / 'df_ciiu_1996.parquet')

df_ciiu_mio = pd.read_parquet(temp / 'pib' / 'df_ciiu_mio.parquet').reset_index()

df_hat = pd.read_parquet(processed / 'df_hat_complex.parquet')

df_mio_va = pd.read_excel(raw / 'pib' / 'VA' / 'MIO-VA.xlsx', sheet_name=None)
df_mio = df_mio_va['MIO']
df_mio['CIIU MIO'] = df_mio['CIIU MIO'].astype(str)

df_va = (
    pd.read_parquet(temp / 'pib' / 'df_va.parquet').reset_index(names='CIIU')
    .merge(
        df_mio_va['VA'],
        on='CIIU',
        how='left'
    )
    .groupby('CIIU VA')[['Total producción','Valor agregado']].sum().reset_index()
)
df_va['VA%'] = df_va['Valor agregado'] / df_va['Total producción']
df_va = df_va[['CIIU VA','VA%']]

B = df_multiplicadores.copy()

### Deflactores

Encadena la inflación anual para llevar los valores a precios constantes de 2015, tanto desde
2019 como desde 2024.

In [5]:
pi_16 = 0.0575
pi_17 = 0.0409
pi_18 = 0.0318
pi_19 = 0.0380
den_k = (1+pi_16)*(1+pi_17)*(1+pi_18)*(1+pi_19)
deflactor_2019_15 = 1/den_k

In [6]:
pi_16 = 0.0575
pi_17 = 0.0409
pi_18 = 0.0318
pi_19 = 0.0380
pi_20 = 0.0161
pi_21 = 0.0562
pi_22 = 0.1312
pi_23 = 0.0923
pi_24 = 0.0520
den_k = (1+pi_16)*(1+pi_17)*(1+pi_18)*(1+pi_19)*(1+pi_20)*(1+pi_21)*(1+pi_22)*(1+pi_23)*(1+pi_24)
deflactor_2024_15 = 1/den_k

### Base de comparación

Expresa en pesos constantes el cambio observado en exportaciones no tradicionales de 2024, que
sirve de referencia para la tasa de crecimiento implícita.

In [7]:
delta_expo_no_trad_24_pesos_24 = (2_141_455_655 * 4_072.59)
delta_expo_no_trad_24_pesos_15 = delta_expo_no_trad_24_pesos_24 *deflactor_2024_15

### Bootstrap del efecto sobre el producto interno bruto

Remuestrea mil veces el panel de impactos conservando una fracción de los productos, que
representa la proporción de ellos que efectivamente consolida un auge exportador. Para cada
réplica traduce el choque exportador a producción por rama mediante la inversa de Leontief, y
de ahí a valor agregado.

El notebook original repetía este bloque completo una vez por escenario; aquí se define una sola
vez como función del porcentaje.

In [8]:
def correr_bootstrap(porcentaje, df_panel=None, hs6_prioritarios=None):
    """Remuestrea el panel de impactos y devuelve las mil replicas de PIB, exportaciones y crecimiento.

    Parametros
    ----------
    porcentaje : float
        Porcentaje del panel que se remuestrea en cada iteracion.
    df_panel : pd.DataFrame, opcional
        Panel sobre el que se remuestrea. Por defecto el panel completo `df_hat`;
        para segmentar por tipo de oportunidad se pasa el subconjunto de la categoria.
    hs6_prioritarios : set, opcional
        Productos que pertenecen a industrias priorizadas. Si se pasa, se devuelve
        ademas la serie de exportaciones restringida a esos productos.
    """
    df_panel = df_hat if df_panel is None else df_panel

    resultados_pib = list()
    resultados_exp = list()
    resultados_g_exp = list()
    resultados_exp_prior = list()

    for b in range(1000):
        df_submuestra = resample_percent_df(df_panel, porcentaje, random_state=b)

        expos_choque = np.mean(df_submuestra['Delta_x_hat'].sum()) * 1000 * 3_281.09 * deflactor_2019_15
        resultados_exp.append(expos_choque)

        if hs6_prioritarios is not None:
            df_prioritarios = df_submuestra[df_submuestra['hs6_1996'].isin(hs6_prioritarios)]
            resultados_exp_prior.append(
                df_prioritarios['Delta_x_hat'].sum() * 1000 * 3_281.09 * deflactor_2019_15
            )

        g_expo = (expos_choque / delta_expo_no_trad_24_pesos_15)**(1/20)-1
        resultados_g_exp.append(g_expo)

        ## Datos de Choque
        df_choque = (
            df_submuestra
            .merge(df_ciiu[['HS6', 'CIIU']], left_on='hs6_1996', right_on='HS6', how='left')
            [['CIIU','hs6_1996','Delta_x_hat','d_iso3','subregion_d']]
            .groupby('CIIU')['Delta_x_hat'].sum()
            .map(lambda x: round(x * 1000 * 3_281.09 * deflactor_2019_15))
            .reset_index()
            .merge(df_ciiu_mio, on='CIIU', how='left')
            .drop(columns='CIIU')
            .groupby('CIIU MIO')['Delta_x_hat'].sum()
        )

    
        ## Vector de Choque
        Y = (
            df_multiplicadores
            [['A0101']]
            .drop(columns=['A0101'])
            .merge(df_choque, left_index=True, right_index=True,how='left',)
            .fillna(0)
        )
        ## IO 
        df_semifinal = B.dot(Y)
        ##
        df_mio_total = (
            df_mio
            .merge(
                df_semifinal.reset_index(), left_on='CIIU MIO', right_on='CIIU', how='left')
            .groupby('CIIU VA')['Delta_x_hat'].sum()
            .reset_index()
        )
        ##
        df_total_pib = df_mio_total.merge(df_va, on='CIIU VA', how='left')
        df_total_pib['Delta PIB'] = df_total_pib['Delta_x_hat'] * df_total_pib['VA%']

        resultados_pib.append(df_total_pib['Delta PIB'].sum())

    return resultados_pib, resultados_exp, resultados_g_exp, resultados_exp_prior

### Escenarios

Corre el bootstrap para los dos escenarios de tasa de éxito y exporta los intervalos de
confianza junto con la serie anual.

In [9]:
# tabla de parametros: porcentaje de exito exportador y archivo de salida.
# el original solo escribia estos dos escenarios; el de 20% del libro de resultados
# se armo a mano por fuera de la cadena. Ver tools/markdown/pendientes.md
parametros_escenario = [
    (10, 'resultados_10.xlsx'),
    (15, 'resultados_15.xlsx'),
]

tables.mkdir(parents=True, exist_ok=True)
for porcentaje, salida in parametros_escenario:
    res_pib, res_exp, res_g_exp, _ = correr_bootstrap(porcentaje)

    df_numeros = pd.DataFrame({
        'Delta PIB': [np.mean(res_pib)],
        'Delta PIB lower bound': [np.percentile(res_pib, 2.5)],
        'Delta PIB upper bound': [np.percentile(res_pib, 97.5)],
        'Crecimiento Promedio': [np.mean(res_g_exp)],
        'Crecimiento lower bound': [np.percentile(res_g_exp, 2.5)],
        'Crecimiento upper bound': [np.percentile(res_g_exp, 97.5)],
        'Delta Exportaciones': [np.mean(res_exp)],
        'Delta Exportaciones lower bound': [np.percentile(res_exp, 2.5)],
        'Delta Exportaciones upper bound': [np.percentile(res_exp, 97.5)]
    }).T

    df_anos = pd.DataFrame({'Año': list(range(2025, 2046))}).set_index('Año')

    with pd.ExcelWriter(tables / salida) as writer:
        df_numeros.to_excel(writer, sheet_name='Números')
        df_anos.to_excel(writer, sheet_name='Series')

    print(f"escenario {porcentaje}%: Delta PIB promedio = {np.mean(res_pib):,.0f}")

escenario 10%: Delta PIB promedio = 187,970,532,326,441


escenario 15%: Delta PIB promedio = 281,761,941,182,789


### Segmentación por tipo de oportunidad

El método del proyecto tiene tres pasos y hasta aquí solo se han hecho el primero y el tercero:
remuestrear y calcular métricas. Falta el segundo, segmentar la submuestra por tipo de oportunidad,
que es lo que produce las tablas por categoría del entregable.

La clasificación está en `df_productos.parquet`, que codifica las categorías múltiples como
etiquetas compuestas unidas por `" - "`: un producto marcado
`Oportunidades a profundizar - Oportunidades de largo alcance` pertenece a las dos. Al expandirlas,
las 809 filas se vuelven 829 asignaciones sobre 809 productos únicos, de modo que **las categorías
se solapan y sus totales no suman el panel**.

Los sectores prioritarios son las industrias de la hoja `Industrias priorizadas 4D C` del archivo
final de productos.

In [10]:
df_tipo_oportunidad = (
    pd.read_parquet(temp / 'exportaciones' / 'df_productos.parquet')
    .assign(tipo_oportunidad=lambda d: d['tipo_oportunidad'].str.split(' - '))
    .explode('tipo_oportunidad')
    .reset_index(drop=True)
)

ciiu_prioritarios = set(
    pd.read_excel(raw / 'exportaciones' / 'productos' / '2 - Output Final 040725 V02 - EXT.xlsx',
                  sheet_name='Industrias priorizadas 4D C')['CIIU']
    .astype(str).str.zfill(4)
)

df_ciiu_hs6 = df_ciiu[['HS6', 'CIIU']].copy()
df_ciiu_hs6['CIIU'] = df_ciiu_hs6['CIIU'].astype(str).str.zfill(4)
hs6_prioritarios = set(df_ciiu_hs6[df_ciiu_hs6['CIIU'].isin(ciiu_prioritarios)]['HS6'])

print(df_tipo_oportunidad['tipo_oportunidad'].value_counts().to_string())
print(f"\nProductos unicos: {df_tipo_oportunidad['hs6_1996'].nunique()}")
print(f"Asignaciones:     {len(df_tipo_oportunidad)}")
print(f"Industrias priorizadas: {len(ciiu_prioritarios)} CIIU, {len(hs6_prioritarios)} productos")

tipo_oportunidad
Oportunidades a profundizar       523
Oportunidades de largo alcance    205
Oportunidades a la vista          101

Productos unicos: 809
Asignaciones:     829
Industrias priorizadas: 34 CIIU, 2020 productos


### Resultados por tipo de oportunidad

Para cada categoría se corre el bootstrap sobre su propio subconjunto del panel y se reportan
exportaciones totales, exportaciones en sectores prioritarios y producto interno bruto, con su
media y sus percentiles 2,5 y 97,5.

El escenario es el base, el del 15%.

In [11]:
escenario_base = 15

resultados_categoria = list()
for categoria in ['Oportunidades a profundizar', 'Oportunidades a la vista', 'Oportunidades de largo alcance']:
    hs6_categoria = set(
        df_tipo_oportunidad.query("tipo_oportunidad == @categoria")['hs6_1996']
    )
    df_categoria = df_hat[df_hat['hs6_1996'].isin(hs6_categoria)]

    res_pib, res_exp, res_g_exp, res_exp_prior = correr_bootstrap(
        escenario_base, df_panel=df_categoria, hs6_prioritarios=hs6_prioritarios
    )

    resultados_categoria.append({
        'Tipo de oportunidad': categoria,
        'Productos': len(hs6_categoria),
        'Exportaciones': np.mean(res_exp),
        'Exportaciones lower bound': np.percentile(res_exp, 2.5),
        'Exportaciones upper bound': np.percentile(res_exp, 97.5),
        'Exportaciones prioritarias': np.mean(res_exp_prior),
        'Exportaciones prioritarias lower bound': np.percentile(res_exp_prior, 2.5),
        'Exportaciones prioritarias upper bound': np.percentile(res_exp_prior, 97.5),
        'Delta PIB': np.mean(res_pib),
        'Delta PIB lower bound': np.percentile(res_pib, 2.5),
        'Delta PIB upper bound': np.percentile(res_pib, 97.5),
    })

df_resultados_categoria = pd.DataFrame(resultados_categoria)

tables.mkdir(parents=True, exist_ok=True)
df_resultados_categoria.to_excel(tables / 'resultados_por_tipo_oportunidad.xlsx', index=False)

df_resultados_categoria

,Tipo de oportunidad,Productos,Exportaciones,Exportaciones lower bound,Exportaciones upper bound,Exportaciones prioritarias,Exportaciones prioritarias lower bound,Exportaciones prioritarias upper bound,Delta PIB,Delta PIB lower bound,Delta PIB upper bound
0,Oportunidades a profundizar,523,"225,963,293,660,079.50","208,719,794,441,301.38","243,291,100,497,171.38","78,775,653,217,760.70","69,199,936,853,904.84","88,585,427,311,073.25","209,328,139,153,203.50","193,299,817,787,923.00","225,761,591,793,545.38"
1,Oportunidades a la vista,101,"30,386,075,699,683.81","25,560,651,998,016.77","36,026,547,522,058.96","13,796,649,659,953.52","11,452,329,085,279.71","16,324,800,041,475.83","28,190,506,418,551.07","23,710,105,353,195.73","33,418,193,606,591.66"
2,Oportunidades de largo alcance,205,"57,215,260,679,561.38","50,579,705,441,860.80","65,049,278,450,194.05","26,100,214,835,176.49","22,297,977,975,564.88","30,885,565,052,969.71","53,129,400,003,319.76","46,890,393,561,027.84","60,394,872,507,027.12"


### Contraste contra las cifras publicadas

El entregable reporta estas mismas tres categorías, en miles de millones de pesos de 2015. La celda
contrasta lo que produce la cadena contra lo publicado.

El resultado hay que leerlo con cuidado y está documentado en el hallazgo 2.12.1 de la auditoría:
dos de las tres categorías caen dentro del intervalo de confianza publicado y la tercera no.

In [12]:
publicado = pd.DataFrame({
    'Tipo de oportunidad': ['Oportunidades a profundizar', 'Oportunidades a la vista', 'Oportunidades de largo alcance'],
    'Exportaciones publicado': [232_474, 30_885, 97_053],
    'Exportaciones prioritarias publicado': [81_786, 14_049, 36_329],
    'Delta PIB publicado': [214_750, 28_655, 88_565],
})

miles_de_millones = 1e9

df_contraste = df_resultados_categoria.merge(publicado, on='Tipo de oportunidad')
for columna in ['Exportaciones', 'Exportaciones prioritarias', 'Delta PIB']:
    df_contraste[columna] = df_contraste[columna] / miles_de_millones
    df_contraste[f'{columna} razon'] = df_contraste[columna] / df_contraste[f'{columna} publicado']

df_contraste[[
    'Tipo de oportunidad',
    'Exportaciones', 'Exportaciones publicado', 'Exportaciones razon',
    'Exportaciones prioritarias', 'Exportaciones prioritarias publicado',
    'Delta PIB', 'Delta PIB publicado', 'Delta PIB razon',
]]

,Tipo de oportunidad,Exportaciones,Exportaciones publicado,Exportaciones razon,Exportaciones prioritarias,Exportaciones prioritarias publicado,Delta PIB,Delta PIB publicado,Delta PIB razon
0,Oportunidades a profundizar,"225,963.29",232474,0.97,"78,775.65",81786,"209,328.14",214750,0.97
1,Oportunidades a la vista,"30,386.08",30885,0.98,"13,796.65",14049,"28,190.51",28655,0.98
2,Oportunidades de largo alcance,"57,215.26",97053,0.59,"26,100.21",36329,"53,129.40",88565,0.60
